In [1]:
import zipfile
from pathlib import Path

# Path to your uploaded ZIP file (make sure it's in /content)
zip_path = "/content/audio_files.zip"  # << change if path differs

# Output folder where images will be extracted
extract_dir = Path("/content/multimodal_audio/audio")

# Create directory if it doesn’t exist
extract_dir.mkdir(parents=True, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Dataset extracted to: {extract_dir.resolve()}")

✅ Dataset extracted to: /content/multimodal_audio/audio


In [2]:
import zipfile
from pathlib import Path

# Path to your uploaded ZIP file (make sure it's in /content)
zip_path = "/content/Multimodal Captioning Dataset.zip"  # << change if path differs

# Output folder where images will be extracted
extract_dir = Path("/content/Multimodal_Captioning_Dataset")

# Create directory if it doesn’t exist
extract_dir.mkdir(parents=True, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Dataset extracted to: {extract_dir.resolve()}")

✅ Dataset extracted to: /content/Multimodal_Captioning_Dataset


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Audio-only multi-label classifier (wav2vec2 embeddings, 5-fold)
# ============================================================
!pip install -q transformers librosa soundfile

import os, math, numpy as np, pandas as pd, random, time
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import soundfile as sf
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# ----------------- Config -----------------
CSV_PATH   = "/content/clips_with_audio_clean2.csv"
AUDIO_COL  = "audio_path"
LABEL_COL  = "probable_classes"
EMB_DIR    = "/content/wav2vec_audio_embs_5fold"
SR         = 16000
AUDIO_SEC  = 3
FOLDS      = 5
EPOCHS     = 20
BATCH_SIZE = 16
LR         = 3e-4
SEED       = 42

os.makedirs(EMB_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ----------------- Repro -----------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ----------------- Load CSV -----------------
df = pd.read_csv(CSV_PATH)
print("Original rows:", len(df))

# Keep only rows with audio + labels
df = df.dropna(subset=[AUDIO_COL, LABEL_COL]).reset_index(drop=True)
print("Rows after dropna:", len(df))

# OPTIONAL: auto-fix audio paths like /audio/ -> /audio/audio_files/
fixed = 0
for i, p in enumerate(df[AUDIO_COL].tolist()):
    if isinstance(p, str) and ("/audio/sample_" in p) and (not os.path.exists(p)):
        new_p = p.replace("/multimodal_audio/audio/", "/multimodal_audio/audio/audio_files/")
        if os.path.exists(new_p):
            df.at[i, AUDIO_COL] = new_p
            fixed += 1
print(f"Audio paths auto-fixed for {fixed} rows.")

# Filter rows where file actually exists
mask = df[AUDIO_COL].apply(lambda x: isinstance(x,str) and os.path.exists(x))
missing = (~mask).sum()
if missing > 0:
    print("Warning: dropping", missing, "rows with missing audio files.")
df = df[mask].reset_index(drop=True)
print("Rows with valid audio:", len(df))

# ----------------- Build multi-label matrix -----------------
def parse_labels(s):
    if isinstance(s, str):
        return [x.strip() for x in s.split(",") if x.strip() != ""]
    return []

df["labels_list"] = df[LABEL_COL].apply(parse_labels)

# Collect all unique labels
all_labels = sorted({l for lst in df["labels_list"] for l in lst})
label_to_idx = {l:i for i,l in enumerate(all_labels)}
num_labels = len(all_labels)
print("Num distinct symptom labels:", num_labels)
print("Example labels:", all_labels[:15])

# Multi-hot matrix
Y = np.zeros((len(df), num_labels), dtype=np.float32)
for i, lst in enumerate(df["labels_list"]):
    for lab in lst:
        if lab in label_to_idx:
            Y[i, label_to_idx[lab]] = 1.0
print("Multi-label matrix shape:", Y.shape)

# Primary label for stratification = first label in list
primary = [lst[0] if len(lst) > 0 else None for lst in df["labels_list"]]
df["primary"] = primary
# If some rows have no labels, drop them
mask2 = df["primary"].notna()
if (~mask2).sum() > 0:
    print("Dropping rows with empty label list:", (~mask2).sum())
    df = df[mask2].reset_index(drop=True)
    Y = Y[mask2.values]

# Reindex after possible drop
print("Rows after cleaning empties:", len(df))
print("Y shape:", Y.shape)

# Encode primary label for StratifiedKFold
from sklearn.preprocessing import LabelEncoder
le_primary = LabelEncoder()
df["primary_id"] = le_primary.fit_transform(df["primary"].astype(str))

# ----------------- Load wav2vec2 -----------------
print("Loading wav2vec2 processor + model (may download ~350MB)...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec.eval()
emb_dim = wav2vec.config.hidden_size
print("wav2vec hidden size:", emb_dim)

# ----------------- Precompute embeddings -----------------
def extract_emb(idx, wav_path):
    out_path = os.path.join(EMB_DIR, f"emb_{idx}.npy")
    if os.path.exists(out_path):
        return
    try:
        y, sr = sf.read(wav_path, dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        # resample
        if sr != SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)
            sr = SR
        # pad / truncate
        wanted = SR * AUDIO_SEC
        if len(y) < wanted:
            y = np.pad(y, (0, wanted - len(y)))
        else:
            y = y[:wanted]
        # to tensor and wav2vec
        with torch.no_grad():
            inputs = processor(y, sampling_rate=SR, return_tensors="pt", padding=True)
            input_values = inputs.input_values.to(device)  # (1, T)
            out = wav2vec(input_values)
            hid = out.last_hidden_state.mean(dim=1).cpu().numpy()[0]  # (768,)
        np.save(out_path, hid.astype(np.float32))
    except Exception as e:
        print(f"Failed to extract emb for {idx}", e)

print(f"Precomputing wav2vec embeddings into {EMB_DIR} ...")
for i, row in tqdm(df.iterrows(), total=len(df)):
    extract_emb(i, row[AUDIO_COL])
print("Embeddings ready.")

# ----------------- Dataset -----------------
class AudioEmbDataset(Dataset):
    def __init__(self, df, Y, indices):
        self.df = df.reset_index(drop=True)
        self.Y = Y
        self.indices = list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        emb = np.load(os.path.join(EMB_DIR, f"emb_{i}.npy"))
        y = self.Y[i]
        return torch.from_numpy(emb).float(), torch.from_numpy(y).float()

# ----------------- Model -----------------
class AudioMLP(nn.Module):
    def __init__(self, emb_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels)
        )
    def forward(self, x):
        return self.net(x)

# ----------------- Training utils -----------------
def compute_pos_weight(y_train):
    # y_train: (N, L) numpy
    pos = y_train.sum(axis=0)      # (L,)
    neg = y_train.shape[0] - pos
    pos_weight = (neg + 1e-3) / (pos + 1e-3)
    return torch.tensor(pos_weight, dtype=torch.float32)

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

oof_logits = np.zeros_like(Y, dtype=np.float32)

for fold, (tr_idx, vl_idx) in enumerate(skf.split(df.index, df["primary_id"].values), start=1):
    print(f"\n=== FOLD {fold}/{FOLDS} ===")
    tr_idx = np.array(tr_idx)
    vl_idx = np.array(vl_idx)

    y_tr = Y[tr_idx]
    pos_weight = compute_pos_weight(y_tr).to(device)
    print("pos_weight (first 10):", pos_weight[:10])

    tr_ds = AudioEmbDataset(df, Y, tr_idx)
    vl_ds = AudioEmbDataset(df, Y, vl_idx)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = AudioMLP(emb_dim=emb_dim, num_labels=num_labels).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_micro = 0.0
    best_ckpt = f"/content/audio_only_fold{fold}.pth"

    for ep in range(1, EPOCHS+1):
        model.train()
        train_loss = 0.0
        for emb_batch, y_batch in tr_loader:
            emb_batch = emb_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(emb_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= max(1, len(tr_loader))

        # validation
        model.eval()
        all_logits = []
        all_targets = []
        with torch.no_grad():
            for emb_batch, y_batch in vl_loader:
                emb_batch = emb_batch.to(device)
                logits = model(emb_batch)
                all_logits.append(logits.cpu().numpy())
                all_targets.append(y_batch.numpy())
        all_logits = np.concatenate(all_logits, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)

        # fixed threshold 0.5 for per-epoch monitoring
        preds = (1 / (1 + np.exp(-all_logits)) >= 0.5).astype(np.float32)
        micro = f1_score(all_targets, preds, average="micro", zero_division=0)
        macro = f1_score(all_targets, preds, average="macro", zero_division=0)

        print(f"Fold{fold} Ep{ep}/{EPOCHS} | Loss {train_loss:.4f} | micro-F1 {micro:.4f} | macro-F1 {macro:.4f}")
        if micro > best_micro + 1e-4:
            best_micro = micro
            torch.save(model.state_dict(), best_ckpt)
            print(f"  -> Saved best model for fold {fold} (micro-F1={best_micro:.4f})")

    # load best and store OOF logits
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    model.eval()
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    offset = 0
    with torch.no_grad():
        for emb_batch, y_batch in vl_loader:
            emb_batch = emb_batch.to(device)
            logits = model(emb_batch).cpu().numpy()
            bs = logits.shape[0]
            oof_logits[vl_idx[offset:offset+bs]] = logits
            offset += bs

# ----------------- Threshold search on OOF -----------------
y_true = Y
logits = oof_logits
probs = 1 / (1 + np.exp(-logits))

best_thr = 0.5
best_micro = 0.0
best_macro = 0.0
for thr in np.linspace(0.3, 0.8, 21):
    preds = (probs >= thr).astype(np.float32)
    micro = f1_score(y_true, preds, average="micro", zero_division=0)
    macro = f1_score(y_true, preds, average="macro", zero_division=0)
    if micro > best_micro:
        best_micro = micro
        best_macro = macro
        best_thr = thr

print(f"\nBest global threshold by OOF micro-F1: {best_thr:.2f}")
print(f"Final OOF micro-F1: {best_micro:.4f} | macro-F1: {best_macro:.4f}")

np.save("/content/audio_only_oof_logits.npy", oof_logits)
print("Saved OOF logits to /content/audio_only_oof_logits.npy")
print("Done.")


Device: cpu
Original rows: 404
Rows after dropna: 404
Audio paths auto-fixed for 0 rows.
Rows with valid audio: 404
Num distinct symptom labels: 18
Example labels: ['cyanosis', 'dry scalp', 'edema', 'eye inflamation', 'eye redness', 'foot swelling', 'hand lump', 'itichy eyelid', 'knee swelling', 'lip swelling', 'mouth ulcer', 'neck swelling', 'skin dryness', 'skin growth', 'skin irritation']
Multi-label matrix shape: (404, 18)
Rows after cleaning empties: 404
Y shape: (404, 18)
Loading wav2vec2 processor + model (may download ~350MB)...


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


wav2vec hidden size: 768
Precomputing wav2vec embeddings into /content/wav2vec_audio_embs_5fold ...


100%|██████████| 404/404 [05:36<00:00,  1.20it/s]

Embeddings ready.

=== FOLD 1/5 ===
pos_weight (first 10): tensor([ 0.9816, 31.2970,  4.2950, 10.5354, 14.3803, 16.9436, 14.3803, 19.1864,
        25.9146,  4.2096])


Fold1 Ep1/20 | Loss 1.2438 | micro-F1 0.2562 | macro-F1 0.1446
  -> Saved best model for fold 1 (micro-F1=0.2562)
Fold1 Ep2/20 | Loss 1.0767 | micro-F1 0.2895 | macro-F1 0.2029
  -> Saved best model for fold 1 (micro-F1=0.2895)
Fold1 Ep3/20 | Loss 1.0279 | micro-F1 0.3072 | macro-F1 0.2459
  -> Saved best model for fold 1 (micro-F1=0.3072)
Fold1 Ep4/20 | Loss 0.9853 | micro-F1 0.2970 | macro-F1 0.2631
Fold1 Ep5/20 | Loss 0.9212 | micro-F1 0.3474 | macro-F1 0.2934
  -> Saved best model for fold 1 (micro-F1=0.3474)
Fold1 Ep6/20 | Loss 0.8873 | micro-F1 0.3735 | macro-F1 0.3042
  -> Saved best model for fold 1 (micro-F1=0.3735)
Fold1 Ep7/20 | Loss 0.8628 | micro-F1 0.3420 | macro-F1 0.2881
Fold1 Ep8/20 | Loss 0.8269 | micro-F1 0.3059 | macro-F1 0.2762
Fold1 Ep9/20 | Loss 0.8031 | micro-F1 0.3848 | macro-F1 0.3146
  -> Saved best model for fold 1 (micro-F1=0.3848)
Fold1 Ep10/20 | Loss 0.7711 | micro-F1 0.4184 | macro-F1 0.3096
  -> Saved best model for fold 1 (micro-F1=0.4184)
Fold1 Ep11/2

In [3]:
# ================== 4) TEXT-ONLY BASELINE (TF-IDF + Logistic Regression) K=5 ==================
!pip install -q scikit-learn

import numpy as np
import pandas as pd
import os, random

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

# -------- CONFIG --------
CSV_PATH  = "/content/clips_with_audio_clean2.csv"
TEXT_COL  = "Updated_Question"   # or "Question_summ"
LABEL_COL = "probable_classes"
FOLDS     = 5
SEED      = 42

random.seed(SEED)
np.random.seed(SEED)

df = pd.read_csv(CSV_PATH)
print("Original rows:", len(df))

mask = df[TEXT_COL].notna() & df[LABEL_COL].notna()
df = df[mask].reset_index(drop=True)
print("Rows after filter:", len(df))

# labels
all_labels = set()
for s in df[LABEL_COL]:
    for t in str(s).split(","):
        t = t.strip()
        if t:
            all_labels.add(t)
all_labels = sorted(list(all_labels))
label_to_idx = {l:i for i,l in enumerate(all_labels)}
L = len(all_labels)
print("Num labels:", L)

Y = np.zeros((len(df), L), dtype=np.float32)
for i, s in enumerate(df[LABEL_COL]):
    for t in str(s).split(","):
        t = t.strip()
        if t in label_to_idx:
            Y[i, label_to_idx[t]] = 1.0

primary = Y.argmax(axis=1)
texts = df[TEXT_COL].fillna("").astype(str).values

# global TF-IDF (fit once, reuse across folds)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_all = tfidf.fit_transform(texts)

def eval_f1(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(np.float32)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return micro, macro

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_prob = np.zeros((len(df), L), dtype=np.float32)

for fold, (tr_idx, vl_idx) in enumerate(skf.split(df.index, primary), start=1):
    print(f"\n=== FOLD {fold}/{FOLDS} ===")
    X_tr, X_vl = X_all[tr_idx], X_all[vl_idx]
    Y_tr, Y_vl = Y[tr_idx], Y[vl_idx]

    clf = OneVsRestClassifier(
        LogisticRegression(max_iter=300, C=2.0, class_weight="balanced", n_jobs=-1)
    )
    clf.fit(X_tr, Y_tr)

    prob = clf.predict_proba(X_vl)
    oof_prob[vl_idx] = prob
    micro, macro = eval_f1(Y_vl, prob, thr=0.5)
    print(f"Fold{fold} | micro-F1 {micro:.4f} | macro-F1 {macro:.4f}")

# global threshold
best_thr, best_micro = 0.5, 0.0
for thr in np.linspace(0.3,0.8,11):
    micro, _ = eval_f1(Y, oof_prob, thr=thr)
    if micro > best_micro:
        best_micro, best_thr = micro, thr

micro, macro = eval_f1(Y, oof_prob, thr=best_thr)
print(f"\nText-only TF-IDF+LR best thr={best_thr:.2f} | micro-F1={micro:.4f} | macro-F1={macro:.4f}")
print("Done (text-only 5-fold).")


Original rows: 404
Rows after filter: 404
Num labels: 18

=== FOLD 1/5 ===


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Fold1 | micro-F1 0.7012 | macro-F1 0.5263

=== FOLD 2/5 ===
Fold2 | micro-F1 0.7717 | macro-F1 0.5201

=== FOLD 3/5 ===
Fold3 | micro-F1 0.6893 | macro-F1 0.4691

=== FOLD 4/5 ===
Fold4 | micro-F1 0.6904 | macro-F1 0.4102

=== FOLD 5/5 ===
Fold5 | micro-F1 0.6714 | macro-F1 0.4703

Text-only TF-IDF+LR best thr=0.45 | micro-F1=0.7222 | macro-F1=0.5504
Done (text-only 5-fold).


In [ ]:
# ================== Tri-Modal (Text + Image + Audio) Fusion – 5-Fold ==================
# Uses:
#   - TF-IDF text features
#   - wav2vec2 mean-pooled audio embeddings
#   - EfficientNet-B3 image features
# Task: multi-label classification over 18 symptom labels

!pip install -q timm transformers librosa soundfile scikit-learn

import os, math, random, time, gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer

import soundfile as sf
import librosa
import timm
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# -------------------- Paths & constants --------------------
CSV_PATH   = "/content/clips_with_audio_clean2.csv"
TEXT_COL_CANDIDATES = ["Updated_Question", "Question_summ"]
IMG_COL_CANDIDATES  = ["Image_path_final", "Image_path"]
AUDIO_COL  = "audio_path"
LABEL_COL  = "probable_classes"

EMB_DIR    = "/content/wav2vec_tri_embs"
os.makedirs(EMB_DIR, exist_ok=True)

IMG_SIZE   = 300
BATCH_SIZE = 4
EPOCHS     = 20
FOLDS      = 5
SEED       = 42
TARGET_SR  = 16000
AUDIO_SEC  = 3

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

# -------------------- Load CSV --------------------
df = pd.read_csv(CSV_PATH)
print("Original rows:", len(df))

# pick text column
TEXT_COL = None
for c in TEXT_COL_CANDIDATES:
    if c in df.columns:
        TEXT_COL = c
        break
if TEXT_COL is None:
    raise ValueError(f"None of {TEXT_COL_CANDIDATES} found in CSV.")

# pick image column
IMG_COL = None
for c in IMG_COL_CANDIDATES:
    if c in df.columns:
        IMG_COL = c
        break
if IMG_COL is None:
    raise ValueError(f"None of {IMG_COL_CANDIDATES} found in CSV.")

# basic filter: need image, audio, label, text
mask = df[IMG_COL].notna() & df[AUDIO_COL].notna() & df[LABEL_COL].notna() & df[TEXT_COL].notna()
df = df[mask].reset_index(drop=True)
print("Rows after filter:", len(df))

# -------------------- Build multi-label matrix --------------------
# probable_classes is something like: "skin rash,skin irritation,cyanosis"
raw_labels = df[LABEL_COL].astype(str).tolist()
all_labels = set()
for s in raw_labels:
    for tok in s.split(","):
        tok = tok.strip()
        if tok:
            all_labels.add(tok)
all_labels = sorted(list(all_labels))
label_to_idx = {lab:i for i,lab in enumerate(all_labels)}
print("Num distinct labels:", len(all_labels))
print("Example labels:", all_labels[:15])
num_labels = len(all_labels)

Y = np.zeros((len(df), num_labels), dtype=np.float32)
for i, s in enumerate(raw_labels):
    for tok in s.split(","):
        tok = tok.strip()
        if tok in label_to_idx:
            Y[i, label_to_idx[tok]] = 1.0

print("Multi-label matrix shape:", Y.shape)

# add row_id for stable mapping to embeddings & tf-idf
df["row_id"] = np.arange(len(df))

# -------------------- Text: TF-IDF features --------------------
texts = df[TEXT_COL].fillna("").astype(str).tolist()
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=2)
X_text_sparse = tfidf.fit_transform(texts)   # (N, D_text)
text_dim = X_text_sparse.shape[1]
print("TF-IDF dim:", text_dim)

# -------------------- Audio: wav2vec2 embedding precomputation --------------------
print("Loading wav2vec2 processor + model (may download ~350MB)...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device).eval()
emb_dim = wav2vec_model.config.hidden_size
print("wav2vec hidden size:", emb_dim)

def extract_wav2vec_emb(row_id, audio_path):
    """
    Save emb_{row_id}.npy in EMB_DIR if not exists.
    """
    out_path = os.path.join(EMB_DIR, f"emb_{row_id}.npy")
    if os.path.exists(out_path):
        return
    try:
        y, sr = sf.read(audio_path, dtype="float32")
    except Exception as e:
        # try librosa as backup
        try:
            y, sr = librosa.load(audio_path, sr=None, mono=False)
            y = y.astype("float32")
        except Exception as e2:
            print(f"  [WARN] Skipping audio {audio_path}: {e} / {e2}")
            # create zero emb
            np.save(out_path, np.zeros(emb_dim, dtype=np.float32))
            return

    if y.ndim > 1:
        y = np.mean(y, axis=1)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    needed = TARGET_SR * AUDIO_SEC
    if len(y) < needed:
        y = np.pad(y, (0, needed - len(y)))
    else:
        y = y[:needed]

    with torch.no_grad():
        inp = processor(y, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
        input_values = inp["input_values"].to(device)   # (1, T)
        out = wav2vec_model(input_values)
        hid = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32)
    np.save(out_path, hid)

print(f"Precomputing wav2vec embeddings into {EMB_DIR} ...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    rid = int(row["row_id"])
    ap = row[AUDIO_COL]
    extract_wav2vec_emb(rid, ap)
print("Embeddings ready.")

# -------------------- Image transforms --------------------
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(0.12, 0.12, 0.08, 0.04),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# -------------------- Dataset --------------------
class TriModalDataset(Dataset):
    def __init__(self, df_sub, X_text_sparse, Y_full, img_transform, emb_dir):
        self.df_sub = df_sub.reset_index(drop=True)
        self.row_ids = self.df_sub["row_id"].values.astype(int)
        self.X_text_sparse = X_text_sparse
        self.Y_full = Y_full
        self.img_transform = img_transform
        self.emb_dir = emb_dir

    def __len__(self):
        return len(self.df_sub)

    def __getitem__(self, idx):
        row = self.df_sub.iloc[idx]
        rid = self.row_ids[idx]

        # image
        img_path = row[IMG_COL]
        img = Image.open(img_path).convert("RGB")
        img = self.img_transform(img)

        # text vec (dense float32)
        x_text = self.X_text_sparse[rid].toarray().astype(np.float32).squeeze()  # (text_dim,)

        # audio emb
        emb_path = os.path.join(self.emb_dir, f"emb_{rid}.npy")
        emb = np.load(emb_path).astype(np.float32)  # (emb_dim,)

        # label
        y = self.Y_full[rid]

        return img, x_text, emb, y

# -------------------- Model --------------------
class ImgEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b3", pretrained=True)
        if hasattr(self.backbone, "classifier"):
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
            self.out_dim = in_features
        elif hasattr(self.backbone, "fc"):
            in_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
            self.out_dim = in_features
        else:
            # fallback: run dummy
            dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE)
            out = self.backbone(dummy)
            self.out_dim = out.shape[1]

    def forward(self, x):
        return self.backbone(x)

class TriModalNet(nn.Module):
    def __init__(self, text_dim, audio_dim, n_classes):
        super().__init__()
        self.img_enc = ImgEncoder()
        img_dim = self.img_enc.out_dim

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.3)
        )
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.3)
        )

        fusion_dim = img_dim + 512 + 512
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.LayerNorm(1024),
            nn.Dropout(0.4),
            nn.Linear(1024, n_classes)
        )

    def forward(self, img, txt, aud):
        img_feat = self.img_enc(img)          # (B, img_dim)
        txt_feat = self.text_proj(txt)        # (B, 512)
        aud_feat = self.audio_proj(aud)       # (B, 512)
        x = torch.cat([img_feat, txt_feat, aud_feat], dim=1)
        logits = self.head(x)
        return logits

# -------------------- Metrics --------------------
def f1_micro_macro(y_true, y_prob, thr=0.5, eps=1e-9):
    """
    y_true, y_prob: numpy arrays (N, C)
    """
    y_pred = (y_prob >= thr).astype(np.int32)

    # micro
    tp = np.logical_and(y_true==1, y_pred==1).sum()
    fp = np.logical_and(y_true==0, y_pred==1).sum()
    fn = np.logical_and(y_true==1, y_pred==0).sum()
    micro_prec = tp / (tp + fp + eps)
    micro_rec  = tp / (tp + fn + eps)
    micro_f1   = 2 * micro_prec * micro_rec / (micro_prec + micro_rec + eps)

    # macro
    C = y_true.shape[1]
    f1s = []
    for c in range(C):
        yt = y_true[:,c]
        yp = y_pred[:,c]
        tp_c = np.logical_and(yt==1, yp==1).sum()
        fp_c = np.logical_and(yt==0, yp==1).sum()
        fn_c = np.logical_and(yt==1, yp==0).sum()
        if tp_c + fp_c + fn_c == 0:
            continue
        prec_c = tp_c / (tp_c + fp_c + eps)
        rec_c  = tp_c / (tp_c + fn_c + eps)
        f1_c   = 2 * prec_c * rec_c / (prec_c + rec_c + eps)
        f1s.append(f1_c)
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0

    return micro_f1, macro_f1

# -------------------- 5-Fold training --------------------
indices = np.arange(len(df))
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

oof_logits = np.zeros((len(df), num_labels), dtype=np.float32)

for fold, (tr_idx, vl_idx) in enumerate(kf.split(indices), start=1):
    print(f"\n=== FOLD {fold}/{FOLDS} ===")

    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    vl_df = df.iloc[vl_idx].reset_index(drop=True)

    tr_ds = TriModalDataset(tr_df, X_text_sparse, Y, train_tf, EMB_DIR)
    vl_ds = TriModalDataset(vl_df, X_text_sparse, Y, eval_tf, EMB_DIR)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # pos_weight from training labels
    y_tr = Y[tr_idx]  # (N_tr, C)
    pos = y_tr.sum(axis=0) + 1e-6
    neg = y_tr.shape[0] - pos + 1e-6
    pos_weight = torch.tensor(neg / pos, dtype=torch.float32, device=device)
    print("pos_weight (first 10):", pos_weight[:10])

    model = TriModalNet(text_dim=text_dim, audio_dim=emb_dim, n_classes=num_labels).to(device)

    # Different LR for image encoder vs fusion head
    img_params   = [p for p in model.img_enc.parameters() if p.requires_grad]
    other_params = [p for n,p in model.named_parameters() if not n.startswith("img_enc.") and p.requires_grad]

    optimizer = optim.AdamW([
        {"params": img_params, "lr": 1e-4},
        {"params": other_params, "lr": 3e-4},
    ], weight_decay=1e-5)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_micro = 0.0
    best_path = f"/content/trimodal_fold{fold}.pth"

    for epoch in range(1, EPOCHS+1):
        model.train()
        running_loss = 0.0
        for imgs, x_text, emb, y in tr_loader:
            imgs = imgs.to(device)
            x_text = x_text.to(device)
            emb = emb.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(imgs, x_text, emb)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item()

        # validation
        model.eval()
        all_true = []
        all_prob = []
        with torch.no_grad():
            for imgs, x_text, emb, y in vl_loader:
                imgs = imgs.to(device)
                x_text = x_text.to(device)
                emb = emb.to(device)
                logits = model(imgs, x_text, emb)
                probs = torch.sigmoid(logits).cpu().numpy()
                all_prob.append(probs)
                all_true.append(y.numpy())

        all_true = np.concatenate(all_true, axis=0)
        all_prob = np.concatenate(all_prob, axis=0)

        micro_f1, macro_f1 = f1_micro_macro(all_true, all_prob, thr=0.5)
        print(f"Fold{fold} Ep{epoch}/{EPOCHS} | Loss {running_loss/len(tr_loader):.4f} | micro-F1 {micro_f1:.4f} | macro-F1 {macro_f1:.4f}")

        if micro_f1 > best_micro + 1e-6:
            best_micro = micro_f1
            torch.save(model.state_dict(), best_path)
            print(f"  -> Saved best model for fold {fold} (micro-F1={best_micro:.4f})")

    # load best for OOF
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    # fill OOF logits for this fold
    vl_ds = TriModalDataset(vl_df, X_text_sparse, Y, eval_tf, EMB_DIR)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    probs_fold = []
    with torch.no_grad():
        for imgs, x_text, emb, y in vl_loader:
            imgs = imgs.to(device)
            x_text = x_text.to(device)
            emb = emb.to(device)
            logits = model(imgs, x_text, emb)
            probs = torch.sigmoid(logits).cpu().numpy()
            probs_fold.append(probs)
    probs_fold = np.concatenate(probs_fold, axis=0)
    oof_logits[vl_idx] = probs_fold
    print(f"Fold {fold} done. Best micro-F1: {best_micro:.4f}")

    # free gpu
    del model, optimizer, tr_loader, vl_loader
    torch.cuda.empty_cache()
    gc.collect()

# -------------------- Global threshold search --------------------
best_thr = 0.5
best_micro = 0.0
best_macro = 0.0
for thr in np.linspace(0.3, 0.8, 11):
    micro_f1, macro_f1 = f1_micro_macro(Y, oof_logits, thr=thr)
    if micro_f1 > best_micro:
        best_micro = micro_f1
        best_macro = macro_f1
        best_thr = thr

print(f"\nBest global threshold: {best_thr:.2f} | micro-F1: {best_micro:.4f} | macro-F1: {best_macro:.4f}")

final_micro, final_macro = f1_micro_macro(Y, oof_logits, thr=best_thr)
print(f"Final OOF micro-F1: {final_micro:.4f} | macro-F1: {final_macro:.4f}")

np.save("/content/trimodal_oof_logits.npy", oof_logits)
print("Saved tri-modal OOF logits to /content/trimodal_oof_logits.npy")
print("Done (Text + Image + Audio, 5-fold).")


DEVICE: cpu
Original rows: 404
Rows after filter: 404
Num distinct labels: 18
Example labels: ['cyanosis', 'dry scalp', 'edema', 'eye inflamation', 'eye redness', 'foot swelling', 'hand lump', 'itichy eyelid', 'knee swelling', 'lip swelling', 'mouth ulcer', 'neck swelling', 'skin dryness', 'skin growth', 'skin irritation']
Multi-label matrix shape: (404, 18)
TF-IDF dim: 2000
Loading wav2vec2 processor + model (may download ~350MB)...


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


wav2vec hidden size: 768
Precomputing wav2vec embeddings into /content/wav2vec_tri_embs ...


100%|██████████| 404/404 [04:27<00:00,  1.51it/s]


Embeddings ready.

=== FOLD 1/5 ===
pos_weight (first 10): tensor([ 0.9816, 25.9167,  4.4746, 10.5357, 14.3810, 19.1875, 15.1500, 20.5333,
        28.3636,  3.7500])


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Fold1 Ep1/20 | Loss 1.3213 | micro-F1 0.5288 | macro-F1 0.3514
  -> Saved best model for fold 1 (micro-F1=0.5288)
Fold1 Ep2/20 | Loss 0.5878 | micro-F1 0.7319 | macro-F1 0.6037
  -> Saved best model for fold 1 (micro-F1=0.7319)
Fold1 Ep3/20 | Loss 0.2438 | micro-F1 0.7662 | macro-F1 0.6816
  -> Saved best model for fold 1 (micro-F1=0.7662)
Fold1 Ep4/20 | Loss 0.1132 | micro-F1 0.7702 | macro-F1 0.6710
  -> Saved best model for fold 1 (micro-F1=0.7702)
Fold1 Ep5/20 | Loss 0.0632 | micro-F1 0.7906 | macro-F1 0.6772
  -> Saved best model for fold 1 (micro-F1=0.7906)
Fold1 Ep6/20 | Loss 0.0342 | micro-F1 0.7828 | macro-F1 0.6232
Fold1 Ep7/20 | Loss 0.0180 | micro-F1 0.8051 | macro-F1 0.7025
  -> Saved best model for fold 1 (micro-F1=0.8051)
Fold1 Ep8/20 | Loss 0.0146 | micro-F1 0.7852 | macro-F1 0.5996
Fold1 Ep9/20 | Loss 0.0087 | micro-F1 0.7907 | macro-F1 0.6449
Fold1 Ep10/20 | Loss 0.0076 | micro-F1 0.8085 | macro-F1 0.6909
  -> Saved best model for fold 1 (micro-F1=0.8085)
Fold1 Ep11/2

In [4]:
# ================== 1) MULTIMODAL (IMAGE + Wav2Vec2) K=5 ==================
!pip install -q timm transformers librosa soundfile scikit-learn

import os, math, random, numpy as np, pandas as pd, time
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

import soundfile as sf
import librosa
import timm
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# ----------------- CONFIG -----------------
CSV_PATH   = "/content/clips_with_audio_clean2.csv"
IMG_COL    = "Image_path_final"
AUDIO_COL  = "audio_path"
LABEL_COL  = "probable_classes"

IMG_SIZE   = 300
BATCH_SIZE = 4
FOLDS      = 5
EPOCHS     = 20
MAX_LR     = 3e-4
SEED       = 42
TARGET_SR  = 16000
AUDIO_SEC  = 3
NUM_WORKERS = 2
EMB_DIR    = "/content/wav2vec_multi_embs"
os.makedirs(EMB_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ----------------- LOAD & PREP LABELS -----------------
df = pd.read_csv(CSV_PATH)
print("Original rows:", len(df))

# fix image paths if needed
def fix_img_path(p):
    if isinstance(p, str) and not os.path.exists(p):
        # common duplication bug
        if "Multimodal Captioning Dataset/Multimodal Captioning Dataset" in p:
            q = p.replace("Multimodal Captioning Dataset/Multimodal Captioning Dataset",
                          "Multimodal_Captioning_Dataset")
            if os.path.exists(q):
                return q
        # if your real layout is different, adjust here
    return p

df[IMG_COL] = df[IMG_COL].apply(fix_img_path)

# fix audio path: /audio/ -> /audio/audio_files/
def fix_audio_path(p):
    if isinstance(p, str) and not os.path.exists(p):
        if "/audio/" in p:
            q = p.replace("/audio/", "/audio/audio_files/")
            if os.path.exists(q):
                return q
    return p

df[AUDIO_COL] = df[AUDIO_COL].apply(fix_audio_path)

# keep only rows with valid paths + probable_classes
mask = df[AUDIO_COL].apply(lambda x: isinstance(x,str) and os.path.exists(x)) & \
       df[IMG_COL].apply(lambda x: isinstance(x,str) and os.path.exists(x)) & \
       df[LABEL_COL].notna()
df = df[mask].reset_index(drop=True)
print("Rows after path+label filter:", len(df))

# build multi-label matrix
all_labels = set()
for s in df[LABEL_COL]:
    for t in str(s).split(","):
        t = t.strip()
        if t:
            all_labels.add(t)

all_labels = sorted(list(all_labels))
label_to_idx = {l:i for i,l in enumerate(all_labels)}
L = len(all_labels)
print("Num distinct labels:", L)
print("Example labels:", all_labels[:15])

Y = np.zeros((len(df), L), dtype=np.float32)
for i, s in enumerate(df[LABEL_COL]):
    for t in str(s).split(","):
        t = t.strip()
        if t in label_to_idx:
            Y[i, label_to_idx[t]] = 1.0

print("Multi-label matrix shape:", Y.shape)

# primary label for stratification (just first non-zero)
primary = Y.argmax(axis=1)

# ----------------- Wav2Vec2: precompute embeddings -----------------
print("Loading wav2vec2 processor + model (may download ~350MB)...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec.eval()
emb_dim = wav2vec.config.hidden_size
print("wav2vec hidden size:", emb_dim)

def extract_emb_for_row(i, row):
    out_path = os.path.join(EMB_DIR, f"emb_{i}.npy")
    if os.path.exists(out_path):
        return
    wav_path = row[AUDIO_COL]
    y, sr = sf.read(wav_path, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR
    wanted = TARGET_SR * AUDIO_SEC
    if len(y) < wanted:
        y = np.pad(y, (0, wanted - len(y)))
    else:
        y = y[:wanted]
    with torch.no_grad():
        inputs = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
        input_values = inputs.input_values.to(device)
        out = wav2vec(input_values)
        hid = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
    np.save(out_path, hid.astype(np.float32))

print(f"Precomputing wav2vec embeddings into {EMB_DIR} ...")
for i, row in tqdm(df.iterrows(), total=len(df)):
    extract_emb_for_row(i, row)
print("Embeddings ready.")

# ----------------- DATASETS -----------------
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(0.12,0.12,0.08,0.04),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class MultiModalDataset(Dataset):
    def __init__(self, df, Y, indices, transform):
        self.df = df.iloc[indices].reset_index(drop=True)
        self.Y = Y[indices]
        self.transform = transform
        self.indices = indices
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        global_idx = self.indices[idx]
        img = Image.open(row[IMG_COL]).convert("RGB")
        img = self.transform(img)
        emb = np.load(os.path.join(EMB_DIR, f"emb_{global_idx}.npy")).astype(np.float32)
        y = self.Y[idx]
        return img, emb, torch.from_numpy(y)

# ----------------- MODEL -----------------
class ImgEncoder(nn.Module):
    def __init__(self, img_size=IMG_SIZE):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b3", pretrained=True, num_classes=0)
    def forward(self, x):
        return self.backbone(x)

class FusionNet(nn.Module):
    def __init__(self, n_labels, emb_dim):
        super().__init__()
        self.img_net = ImgEncoder()
        self.aud_fc = nn.Sequential(
            nn.Linear(emb_dim, 512),
            nn.ReLU(),
            nn.LayerNorm(512),
        )
        # get feature dim
        with torch.no_grad():
            dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE)
            img_feat_dim = self.img_net(dummy).shape[1]
        fusion_dim = img_feat_dim + 512
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, n_labels),
        )
    def forward(self, img, emb):
        img_feat = self.img_net(img)
        aud_feat = self.aud_fc(emb)
        x = torch.cat([img_feat, aud_feat], dim=1)
        return self.head(x)

# ----------------- LOSS & METRICS -----------------
# pos_weight for BCEWithLogits
pos_counts = Y.sum(axis=0) + 1e-6
neg_counts = (len(Y) - pos_counts) + 1e-6
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32)
print("pos_weight (first 10):", pos_weight[:10])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

def evaluate_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(np.float32)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return micro, macro

# ----------------- TRAIN 5-FOLD -----------------
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_logits = np.zeros((len(df), L), dtype=np.float32)

for fold, (tr_idx, vl_idx) in enumerate(skf.split(df.index, primary), start=1):
    print(f"\n=== FOLD {fold}/{FOLDS} ===")
    tr_ds = MultiModalDataset(df, Y, tr_idx, transform=train_tf)
    vl_ds = MultiModalDataset(df, Y, vl_idx, transform=eval_tf)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

    model = FusionNet(n_labels=L, emb_dim=emb_dim).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-5)
    steps_per_epoch = max(1, len(tr_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR,
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS, pct_start=0.3
    )

    best_micro = 0.0
    ckpt_path = f"/content/multimodal_fold{fold}.pth"

    for ep in range(1, EPOCHS+1):
        model.train()
        tot_loss = 0.0
        for imgs, embs, y in tr_loader:
            imgs = imgs.to(device)
            embs = torch.from_numpy(np.stack(embs.numpy())).to(device) if isinstance(embs, torch.Tensor) else torch.tensor(embs).to(device)
            y = y.to(device).float()
            optimizer.zero_grad()
            logits = model(imgs, embs)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            tot_loss += loss.item()

        # validation
        model.eval()
        vl_logits = []
        vl_y = []
        with torch.no_grad():
            for imgs, embs, y in vl_loader:
                imgs = imgs.to(device)
                embs = torch.from_numpy(np.stack(embs.numpy())).to(device) if isinstance(embs, torch.Tensor) else torch.tensor(embs).to(device)
                y = y.numpy()
                logits = model(imgs, embs).cpu().numpy()
                vl_logits.append(logits)
                vl_y.append(y)
        vl_logits = np.vstack(vl_logits)
        vl_y = np.vstack(vl_y)
        prob = 1 / (1 + np.exp(-vl_logits))
        micro, macro = evaluate_metrics(vl_y, prob, thr=0.5)
        print(f"Fold{fold} Ep{ep}/{EPOCHS} | Loss {tot_loss/len(tr_loader):.4f} | micro-F1 {micro:.4f} | macro-F1 {macro:.4f}")
        if micro > best_micro + 1e-4:
            best_micro = micro
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> Saved best model for fold {fold} (micro-F1={best_micro:.4f})")

    # load best & write OOF
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    with torch.no_grad():
        start = 0
        all_logits = []
        for imgs, embs, y in vl_loader:
            imgs = imgs.to(device)
            embs = torch.from_numpy(np.stack(embs.numpy())).to(device) if isinstance(embs, torch.Tensor) else torch.tensor(embs).to(device)
            logits = model(imgs, embs).cpu().numpy()
            all_logits.append(logits)
        all_logits = np.vstack(all_logits)
    oof_logits[vl_idx] = all_logits
    print(f"Fold {fold} done. Best micro-F1: {best_micro:.4f}")

# global threshold search
oof_prob = 1 / (1 + np.exp(-oof_logits))
best_thr = 0.5
best_micro = 0.0
for thr in np.linspace(0.3, 0.8, 11):
    micro, _ = evaluate_metrics(Y, oof_prob, thr=thr)
    if micro > best_micro:
        best_micro = micro
        best_thr = thr

micro, macro = evaluate_metrics(Y, oof_prob, thr=best_thr)
print(f"\nBest global threshold: {best_thr:.2f} | micro-F1: {micro:.4f}")
print(f"Final OOF micro-F1: {micro:.4f} | macro-F1: {macro:.4f}")

# per-label summary
y_pred = (oof_prob >= best_thr).astype(np.float32)
print("\nPer-label metrics (idx, label, P, R, F1, support):")
for i, lab in enumerate(all_labels):
    tp = ((Y[:,i]==1) & (y_pred[:,i]==1)).sum()
    fp = ((Y[:,i]==0) & (y_pred[:,i]==1)).sum()
    fn = ((Y[:,i]==1) & (y_pred[:,i]==0)).sum()
    supp = int(Y[:,i].sum())
    prec = tp / (tp+fp+1e-6)
    rec = tp / (tp+fn+1e-6)
    f1 = 2*prec*rec/(prec+rec+1e-6)
    print(f"{i:2d} {lab:30s} | P={prec:.3f} R={rec:.3f} F1={f1:.3f} supp={supp}")
print("Done. Multimodal 5-fold finished.")


DEVICE: cuda
Original rows: 404
Rows after path+label filter: 404
Num distinct labels: 18
Example labels: ['cyanosis', 'dry scalp', 'edema', 'eye inflamation', 'eye redness', 'foot swelling', 'hand lump', 'itichy eyelid', 'knee swelling', 'lip swelling', 'mouth ulcer', 'neck swelling', 'skin dryness', 'skin growth', 'skin irritation']
Multi-label matrix shape: (404, 18)
Loading wav2vec2 processor + model (may download ~350MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Fold1 Ep1/20 | Loss 1.1593 | micro-F1 0.4554 | macro-F1 0.2317
  -> Saved best model for fold 1 (micro-F1=0.4554)


/tmp/ipython-input-143107572.py:288: RuntimeWarning: overflow encountered in exp
  prob = 1 / (1 + np.exp(-vl_logits))


Fold1 Ep2/20 | Loss 1.1459 | micro-F1 0.4959 | macro-F1 0.1935
  -> Saved best model for fold 1 (micro-F1=0.4959)
Fold1 Ep3/20 | Loss 1.0243 | micro-F1 0.3889 | macro-F1 0.3186


/tmp/ipython-input-143107572.py:288: RuntimeWarning: overflow encountered in exp
  prob = 1 / (1 + np.exp(-vl_logits))


Fold1 Ep4/20 | Loss 1.0157 | micro-F1 0.3558 | macro-F1 0.3264
Fold1 Ep5/20 | Loss 0.9516 | micro-F1 0.4125 | macro-F1 0.3536
Fold1 Ep6/20 | Loss 0.9381 | micro-F1 0.4383 | macro-F1 0.3816
Fold1 Ep7/20 | Loss 0.8987 | micro-F1 0.4122 | macro-F1 0.3623
Fold1 Ep8/20 | Loss 0.8908 | micro-F1 0.4590 | macro-F1 0.4096
Fold1 Ep9/20 | Loss 0.8724 | micro-F1 0.4694 | macro-F1 0.4213
Fold1 Ep10/20 | Loss 0.8176 | micro-F1 0.4662 | macro-F1 0.3812
Fold1 Ep11/20 | Loss 0.7278 | micro-F1 0.5191 | macro-F1 0.4288
  -> Saved best model for fold 1 (micro-F1=0.5191)
Fold1 Ep12/20 | Loss 0.6630 | micro-F1 0.5358 | macro-F1 0.4406
  -> Saved best model for fold 1 (micro-F1=0.5358)
Fold1 Ep13/20 | Loss 0.6224 | micro-F1 0.5307 | macro-F1 0.4449
Fold1 Ep14/20 | Loss 0.5960 | micro-F1 0.5396 | macro-F1 0.4428
  -> Saved best model for fold 1 (micro-F1=0.5396)
Fold1 Ep15/20 | Loss 0.5248 | micro-F1 0.5460 | macro-F1 0.4414
  -> Saved best model for fold 1 (micro-F1=0.5460)
Fold1 Ep16/20 | Loss 0.5380 | micr

In [5]:
# ----------------- SAVE OOF LOGITS + SUMMARY METRICS -----------------
import os
import pandas as pd
import numpy as np

# ✅ Set appropriate filename for this modality combination
SAVE_NAME = "text_image_oof_logits.npy"  # or "audio_image_oof_logits.npy", etc.
np.save(SAVE_NAME, oof_logits)
print(f"✅ Saved OOF logits to {SAVE_NAME}")

# 📊 Append/update entry in summary CSV
summary_path = "/content/analysis_outputs/summary_metrics.csv"
os.makedirs(os.path.dirname(summary_path), exist_ok=True)

# Define the new row
row = {
    "model": SAVE_NAME.replace("_oof_logits.npy", ""),
    "micro_f1": round(micro, 6),
    "macro_f1": round(macro, 6),
    "threshold": round(best_thr, 2),
}

# Update the CSV
if os.path.exists(summary_path):
    df_sum = pd.read_csv(summary_path)
    df_sum = df_sum[df_sum["model"] != row["model"]]  # remove any existing row
    df_sum = pd.concat([df_sum, pd.DataFrame([row])], ignore_index=True)
else:
    df_sum = pd.DataFrame([row])

df_sum.to_csv(summary_path, index=False)
print(f"📊 Summary metrics updated and saved to {summary_path}")


✅ Saved OOF logits to text_image_oof_logits.npy
📊 Summary metrics updated and saved to /content/analysis_outputs/summary_metrics.csv


In [7]:
# Colab-ready single cell (fixed): Text + Audio fusion (TF-IDF + wav2vec2) -- 5-fold multi-label
# Saves: /content/text_audio_oof_logits.npy
# Updates: /content/analysis_outputs/summary_metrics.csv

# ------ Install needed packages (run once) ------
# Note: use iterative-stratification instead of iterstrat
!pip install -q transformers==4.34.0 datasets==2.17.0 iterative-stratification soundfile

# ------ Imports ------
import os, sys, math, json, glob, random
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import KFold

# Attempt to import MultilabelStratifiedKFold from iterative-stratification
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    MLFOLD_AVAILABLE = True
    print("Using MultilabelStratifiedKFold from iterative-stratification.")
except Exception as e:
    MLFOLD_AVAILABLE = False
    print("iterative-stratification import failed:", e)
    print("Falling back to sklearn KFold (note: this is NOT multilabel-stratified).")

# transformers for wav2vec2
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import soundfile as sf

# ------ User-configurable paths & hyperparams ------
CSV_PATH = "/content/clips_with_audio_clean2.csv"   # your dataset CSV
Y_TRUE_PATH = "/content/analysis_outputs/y_true.npy" # (optional) pre-saved y_true (recommended)
OUTPUT_DIR = "/content/analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PRECOMPUTED_EMB_CANDIDATES = [
    "/content/wav2vec_tri_embs.npy",
    "/content/wav2vec_multi_embs.npy",
    "/content/wav2vec_audio_embs_5fold.npy",
    "/content/wav2vec_audio_embs.npy",
    "/content/wav2vec_embs.npy",
    "/content/wav2vec_text_audio_embs.npy",
]

TEXT_COLUMNS_TO_TRY = [
    "Question_summ",
    "Updated_Question",
    "summary_with_updated_question_imagebindcontext",
    "Image_information",
    "Identified_disorder",
    "Question",
    "text",
    "caption",
]

N_FOLDS = 5
SEED = 42
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-4
TFIDF_MAX_FEAT = 2000
HIDDEN = 1024
DROPOUT = 0.3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ------ Helpers ------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

def try_load_npy(paths):
    for p in paths:
        if os.path.exists(p):
            print("Found precomputed embedding file:", p)
            return np.load(p)
    return None

# ------ Load CSV and build y (multi-label matrix) ------
df = pd.read_csv(CSV_PATH)
print("CSV loaded:", CSV_PATH, "shape:", df.shape)
# Attempt to load y_true if present
if os.path.exists(Y_TRUE_PATH):
    y = np.load(Y_TRUE_PATH)
    print("Loaded existing y_true from", Y_TRUE_PATH, "shape:", y.shape)
    label_names = None
else:
    if "probable_classes" in df.columns:
        label_series = df["probable_classes"].fillna("").astype(str).str.replace(";"," , ")
        label_lists = [ [x.strip() for x in s.replace(";",",").split(",") if x.strip()] for s in label_series.tolist() ]
        mlb = MultiLabelBinarizer(sparse_output=False)
        y = mlb.fit_transform(label_lists)
        label_names = mlb.classes_.tolist()
        print("Built y from 'probable_classes' column. Labels:", label_names)
    elif "labels" in df.columns:
        label_series = df["labels"].fillna("").astype(str)
        label_lists = [ [x.strip() for x in s.replace(";",",").split(",") if x.strip()] for s in label_series.tolist() ]
        mlb = MultiLabelBinarizer(sparse_output=False)
        y = mlb.fit_transform(label_lists)
        label_names = mlb.classes_.tolist()
        print("Built y from 'labels' column. Labels:", label_names)
    else:
        possible_label_cols = [c for c in df.columns if df[c].dropna().isin([0,1]).all()]
        if len(possible_label_cols) >= 2:
            y = df[possible_label_cols].fillna(0).astype(int).values
            label_names = possible_label_cols
            print("Built y from binary columns. Labels count:", len(label_names))
        else:
            raise ValueError("Could not find label columns. Please prepare y_true.npy or include a 'probable_classes'/'labels' column.")

N, C = y.shape
print("Dataset size:", N, "labels:", C)

# Save y_true for reproducibility if not present
os.makedirs(os.path.dirname(Y_TRUE_PATH), exist_ok=True)
np.save(Y_TRUE_PATH, y)
print("Saved y_true to", Y_TRUE_PATH)

# ------ Text: select text column and fit TF-IDF ------
text_col = None
for c in TEXT_COLUMNS_TO_TRY:
    if c in df.columns:
        text_col = c
        break
if text_col is None:
    object_cols = [c for c in df.columns if df[c].dtype == object]
    if object_cols:
        text_col = object_cols[0]
    else:
        raise ValueError("No text column found in CSV.")
print("Using text column:", text_col)

texts = df[text_col].fillna("").astype(str).tolist()
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEAT, lowercase=True, stop_words='english')
X_text = tfidf.fit_transform(texts).toarray().astype(np.float32)
print("TF-IDF shape:", X_text.shape)

# ------ Audio: load precomputed embeddings or compute them ------
wav_embs = try_load_npy(PRECOMPUTED_EMB_CANDIDATES)
if wav_embs is None:
    print("No precomputed embeddings found. Computing wav2vec2 embeddings now (this may be slow)...")
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
    model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
    model.eval()
    wav_embs = []
    for i, row in tqdm(df.iterrows(), total=len(df)):
        audio_path = row.get("audio_path", "")
        if not isinstance(audio_path, str) or not audio_path:
            raise FileNotFoundError(f"Row {i} missing 'audio_path'.")
        audio, sr = sf.read(audio_path)
        inputs = processor(audio, sampling_rate=sr, return_tensors="pt", padding=True)
        with torch.no_grad():
            out = model(inputs.input_values.to(DEVICE)).last_hidden_state
            emb = out.mean(dim=1).squeeze().cpu().numpy()
        wav_embs.append(emb)
    wav_embs = np.vstack(wav_embs).astype(np.float32)
    save_path_emb = "/content/wav2vec_text_audio_embs.npy"
    np.save(save_path_emb, wav_embs)
    print("Saved computed wav2vec embeddings to", save_path_emb)

print("Audio embeddings shape:", wav_embs.shape)
assert wav_embs.shape[0] == N, "Embeddings and dataset size mismatch!"

# ------ Dataset / Dataloader ------
class TextAudioDataset(Dataset):
    def __init__(self, text_arr, audio_arr, labels):
        self.text = text_arr
        self.audio = audio_arr
        self.labels = labels
    def __len__(self):
        return len(self.text)
    def __getitem__(self, idx):
        return {
            "text": torch.tensor(self.text[idx], dtype=torch.float32),
            "audio": torch.tensor(self.audio[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ------ Fusion model ------
class FusionNet(nn.Module):
    def __init__(self, dim_text, dim_audio, hidden=HIDDEN, out_dim=C, dropout=DROPOUT):
        super().__init__()
        self.in_dim = dim_text + dim_audio
        self.fc1 = nn.Linear(self.in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden // 2)
        self.out = nn.Linear(hidden // 2, out_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    def forward(self, text, audio):
        x = torch.cat([text, audio], dim=1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.out(x)

# ------ Training / CV loop ------
if MLFOLD_AVAILABLE:
    kf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
else:
    print("WARNING: Multilabel-stratified folds unavailable. Using plain KFold (may produce imbalance across folds).")
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_logits = np.zeros((N, C), dtype=np.float32)
fold_idx = 0

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_text, y) if MLFOLD_AVAILABLE else kf.split(X_text)):
    fold_idx += 1
    print("\n=== FOLD", fold_idx, "/", N_FOLDS, "===")
    if MLFOLD_AVAILABLE:
        X_text_tr, X_text_val = X_text[train_idx], X_text[valid_idx]
        X_audio_tr, X_audio_val = wav_embs[train_idx], wav_embs[valid_idx]
        y_tr, y_val = y[train_idx], y[valid_idx]
    else:
        # sklearn KFold gives indices in different shape
        X_text_tr, X_text_val = X_text[train_idx], X_text[valid_idx]
        X_audio_tr, X_audio_val = wav_embs[train_idx], wav_embs[valid_idx]
        y_tr, y_val = y[train_idx], y[valid_idx]

    train_ds = TextAudioDataset(X_text_tr, X_audio_tr, y_tr)
    valid_ds = TextAudioDataset(X_text_val, X_audio_val, y_val)
    tr_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = FusionNet(dim_text=X_text.shape[1], dim_audio=wav_embs.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    pos_counts = y_tr.sum(axis=0)
    pos_weight_fold = (len(y_tr) - pos_counts) / (pos_counts + 1e-12)
    pos_weight_fold = torch.tensor(pos_weight_fold, dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_fold)

    best_micro = -1.0
    best_state = None

    for epoch in range(1, EPOCHS+1):
        model.train()
        tr_loss = 0.0
        for batch in tr_loader:
            optimizer.zero_grad()
            text_b = batch["text"].to(DEVICE)
            audio_b = batch["audio"].to(DEVICE)
            label_b = batch["label"].to(DEVICE)
            logits = model(text_b, audio_b)
            loss = criterion(logits, label_b)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * text_b.size(0)
        tr_loss = tr_loss / len(train_ds)

        # validation
        model.eval()
        all_logits = []
        all_labels = []
        with torch.no_grad():
            for batch in val_loader:
                text_b = batch["text"].to(DEVICE)
                audio_b = batch["audio"].to(DEVICE)
                logits = model(text_b, audio_b)
                all_logits.append(logits.cpu().numpy())
                all_labels.append(batch["label"].numpy())
        all_logits = np.vstack(all_logits)
        all_labels = np.vstack(all_labels)
        preds = (1 / (1 + np.exp(-all_logits)) ) >= 0.5
        micro = f1_score(all_labels, preds, average='micro', zero_division=0)
        macro = f1_score(all_labels, preds, average='macro', zero_division=0)
        print(f"Fold{fold_idx} Ep{epoch}/{EPOCHS} | Loss {tr_loss:.4f} | micro-F1 {micro:.4f} | macro-F1 {macro:.4f}")
        if micro > best_micro:
            best_micro = micro
            best_state = model.state_dict()

    # compute OOF logits for validation split
    model.load_state_dict(best_state)
    model.eval()
    val_logits = []
    with torch.no_grad():
        for batch in val_loader:
            text_b = batch["text"].to(DEVICE)
            audio_b = batch["audio"].to(DEVICE)
            logits = model(text_b, audio_b)
            val_logits.append(logits.cpu().numpy())
    val_logits = np.vstack(val_logits)
    oof_logits[valid_idx] = val_logits
    print("Fold", fold_idx, "done. Best micro on val (threshold=0.5 monitor):", best_micro)

# Save OOF logits
oof_path = "/content/text_audio_oof_logits.npy"
np.save(oof_path, oof_logits)
print("Saved text+audio OOF logits to", oof_path)

# ------ Threshold tuning on OOF logits & compute final metrics ------
sigmoid = lambda x: 1/(1+np.exp(-x))
probs = sigmoid(oof_logits)

best_thr = None
best_micro = -1.0
best_macro = -1.0
best_thr_for_macro = None

for thr in np.linspace(0.30, 0.90, 61):
    preds = (probs >= thr).astype(int)
    micro = f1_score(y, preds, average='micro', zero_division=0)
    macro = f1_score(y, preds, average='macro', zero_division=0)
    if micro > best_micro:
        best_micro = micro
        best_thr = thr
    if macro > best_macro:
        best_macro = macro
        best_thr_for_macro = thr

print("Best threshold (micro-F1):", best_thr, "micro-F1:", best_micro)
print("Best threshold (macro-F1):", best_thr_for_macro, "macro-F1:", best_macro)

# Save summary_metrics.csv (append if exists)
summary_csv = os.path.join(OUTPUT_DIR, "summary_metrics.csv")
row = {
    "model": "text+audio",
    "micro_f1": float(best_micro),
    "macro_f1": float(best_macro),
    "threshold": float(best_thr)
}
if os.path.exists(summary_csv):
    df_sum = pd.read_csv(summary_csv)
    if row["model"] in df_sum['model'].values:
        df_sum.loc[df_sum['model']==row["model"], ['micro_f1','macro_f1','threshold']] = [row['micro_f1'], row['macro_f1'], row['threshold']]
    else:
        df_sum = pd.concat([df_sum, pd.DataFrame([row])], ignore_index=True)
else:
    df_sum = pd.DataFrame([row])
df_sum.to_csv(summary_csv, index=False)
print("Updated summary CSV at", summary_csv)
print(df_sum)

print("\nDone. Files saved:")
print(" - OOF logits:", oof_path)
print(" - summary metrics:", summary_csv)


ERROR: Cannot install datasets==2.17.0, transformers and transformers==4.34.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
iterative-stratification import failed: No module named 'iterstrat'
Falling back to sklearn KFold (note: this is NOT multilabel-stratified).
Device: cuda
CSV loaded: /content/clips_with_audio_clean2.csv shape: (404, 11)
Built y from 'probable_classes' column. Labels: ['cyanosis', 'dry scalp', 'edema', 'eye inflamation', 'eye redness', 'foot swelling', 'hand lump', 'itichy eyelid', 'knee swelling', 'lip swelling', 'mouth ulcer', 'neck swelling', 'skin dryness', 'skin growth', 'skin irritation', 'skin rash', 'swollen eye', 'swollen tonsil']
Dataset size: 404 labels: 18
Saved y_true to /content/analysis_outputs/y_true.npy
Using text column: Question_summ
TF-IDF shape: (404, 1394)
No precomputed embeddings found. Com

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 404/404 [00:25<00:00, 15.70it/s]


Saved computed wav2vec embeddings to /content/wav2vec_text_audio_embs.npy
Audio embeddings shape: (404, 768)

=== FOLD 1 / 5 ===
Fold1 Ep1/20 | Loss 1.1569 | micro-F1 0.2522 | macro-F1 0.1043
Fold1 Ep2/20 | Loss 1.1548 | micro-F1 0.3636 | macro-F1 0.1321
Fold1 Ep3/20 | Loss 1.1536 | micro-F1 0.3899 | macro-F1 0.1497
Fold1 Ep4/20 | Loss 1.1530 | micro-F1 0.3152 | macro-F1 0.1387
Fold1 Ep5/20 | Loss 1.1515 | micro-F1 0.4171 | macro-F1 0.1847
Fold1 Ep6/20 | Loss 1.1501 | micro-F1 0.3670 | macro-F1 0.2072
Fold1 Ep7/20 | Loss 1.1450 | micro-F1 0.5224 | macro-F1 0.2934
Fold1 Ep8/20 | Loss 1.1398 | micro-F1 0.5699 | macro-F1 0.2959
Fold1 Ep9/20 | Loss 1.1377 | micro-F1 0.5108 | macro-F1 0.3066
Fold1 Ep10/20 | Loss 1.1244 | micro-F1 0.5540 | macro-F1 0.2086
Fold1 Ep11/20 | Loss 1.1153 | micro-F1 0.5491 | macro-F1 0.3088
Fold1 Ep12/20 | Loss 1.0968 | micro-F1 0.5821 | macro-F1 0.3407
Fold1 Ep13/20 | Loss 1.0723 | micro-F1 0.6220 | macro-F1 0.3274
Fold1 Ep14/20 | Loss 1.0362 | micro-F1 0.6318 | 

In [4]:
# ================== 4) TEXT-ONLY BASELINE (TF-IDF + Logistic Regression) K=5 ==================
!pip install -q scikit-learn

import numpy as np
import pandas as pd
import os, random

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

# -------- CONFIG --------
CSV_PATH  = "/content/clips_with_audio_clean2.csv"
TEXT_COL  = "Updated_Question"   # or "Question_summ"
LABEL_COL = "probable_classes"
FOLDS     = 5
SEED      = 42

random.seed(SEED)
np.random.seed(SEED)

df = pd.read_csv(CSV_PATH)
print("Original rows:", len(df))

mask = df[TEXT_COL].notna() & df[LABEL_COL].notna()
df = df[mask].reset_index(drop=True)
print("Rows after filter:", len(df))

# labels
all_labels = set()
for s in df[LABEL_COL]:
    for t in str(s).split(","):
        t = t.strip()
        if t:
            all_labels.add(t)
all_labels = sorted(list(all_labels))
label_to_idx = {l:i for i,l in enumerate(all_labels)}
L = len(all_labels)
print("Num labels:", L)

Y = np.zeros((len(df), L), dtype=np.float32)
for i, s in enumerate(df[LABEL_COL]):
    for t in str(s).split(","):
        t = t.strip()
        if t in label_to_idx:
            Y[i, label_to_idx[t]] = 1.0

primary = Y.argmax(axis=1)
texts = df[TEXT_COL].fillna("").astype(str).values

# global TF-IDF (fit once, reuse across folds)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_all = tfidf.fit_transform(texts)

def eval_f1(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(np.float32)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return micro, macro

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_prob = np.zeros((len(df), L), dtype=np.float32)

for fold, (tr_idx, vl_idx) in enumerate(skf.split(df.index, primary), start=1):
    print(f"\n=== FOLD {fold}/{FOLDS} ===")
    X_tr, X_vl = X_all[tr_idx], X_all[vl_idx]
    Y_tr, Y_vl = Y[tr_idx], Y[vl_idx]

    clf = OneVsRestClassifier(
        LogisticRegression(max_iter=300, C=2.0, class_weight="balanced", n_jobs=-1)
    )
    clf.fit(X_tr, Y_tr)

    prob = clf.predict_proba(X_vl)
    oof_prob[vl_idx] = prob
    micro, macro = eval_f1(Y_vl, prob, thr=0.5)
    print(f"Fold{fold} | micro-F1 {micro:.4f} | macro-F1 {macro:.4f}")

# global threshold
best_thr, best_micro = 0.5, 0.0
for thr in np.linspace(0.3,0.8,11):
    micro, _ = eval_f1(Y, oof_prob, thr=thr)
    if micro > best_micro:
        best_micro, best_thr = micro, thr

micro, macro = eval_f1(Y, oof_prob, thr=best_thr)
print(f"\nText-only TF-IDF+LR best thr={best_thr:.2f} | micro-F1={micro:.4f} | macro-F1={macro:.4f}")

# Save OOF logits
np.save("text_only_oof_logits.npy", oof_prob)
print("✅ Saved OOF logits to text_only_oof_logits.npy")

# Append to summary
summary_path = "/content/analysis_outputs/summary_metrics.csv"
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
row = {
    "model": "text-only",
    "micro_f1": round(micro, 6),
    "macro_f1": round(macro, 6),
    "threshold": round(best_thr, 2),
}
if os.path.exists(summary_path):
    df_sum = pd.read_csv(summary_path)
    df_sum = df_sum[df_sum["model"] != "text-only"]
    df_sum = pd.concat([df_sum, pd.DataFrame([row])], ignore_index=True)
else:
    df_sum = pd.DataFrame([row])

df_sum.to_csv(summary_path, index=False)
print(f"📊 Updated {summary_path}")


Original rows: 404
Rows after filter: 404
Num labels: 18


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(



=== FOLD 1/5 ===
Fold1 | micro-F1 0.7012 | macro-F1 0.5263

=== FOLD 2/5 ===
Fold2 | micro-F1 0.7717 | macro-F1 0.5201

=== FOLD 3/5 ===
Fold3 | micro-F1 0.6893 | macro-F1 0.4691

=== FOLD 4/5 ===
Fold4 | micro-F1 0.6904 | macro-F1 0.4102

=== FOLD 5/5 ===
Fold5 | micro-F1 0.6714 | macro-F1 0.4703

Text-only TF-IDF+LR best thr=0.45 | micro-F1=0.7222 | macro-F1=0.5504
✅ Saved OOF logits to text_only_oof_logits.npy
📊 Updated /content/analysis_outputs/summary_metrics.csv


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv("/content/clips_with_audio_clean2.csv")

# Split multi-label strings into list of labels
df["probable_classes"] = df["probable_classes"].fillna("")
df["probable_classes"] = df["probable_classes"].apply(lambda x: [cls.strip() for cls in x.split(",") if cls.strip()])

mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df["probable_classes"])

# Save
np.save("y_true.npy", y_true)
